# ALTO v1 — Manual validation

Notebook này tách manual review khỏi baseline chính. Nó chỉ đọc `ranking.csv` và `summary_by_sample.csv`, tạo một subset cân bằng theo specification/selection score, sau đó so sánh hard gate và soft ranking với nhãn người chấm.


## 1. Mount Drive và cấu hình


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/alto/v1")
OUTPUT_DIR = PROJECT_DIR / "Output"
RANKING_PATH = OUTPUT_DIR / "ranking.csv"
SAMPLE_SUMMARY_PATH = OUTPUT_DIR / "summary_by_sample.csv"
REVIEW_PATH = OUTPUT_DIR / "manual_review_subset.csv"

MANUAL_SAMPLES_PER_SPEC = 2
MIN_REVIEWED_ROWS = 6


## 2. Tạo hoặc đọc review subset

Mỗi specification chọn background ở các mức selection score trải đều; mỗi background lấy candidate auto-rank tốt nhất và một candidate giữa bảng. File đã tồn tại sẽ không bị ghi đè. Các cột điểm manual dùng thang 1–5; `human_valid` và `human_accept` dùng 0/1.


In [ ]:
ranking_df = pd.read_csv(RANKING_PATH, sep=";", encoding="utf-8-sig")
sample_stats = pd.read_csv(SAMPLE_SUMMARY_PATH, sep=";", encoding="utf-8-sig")

review_payload = {
    "sample_ids": sorted(ranking_df["sample_id"].astype(str).unique().tolist()),
    "seeds": sorted(pd.to_numeric(ranking_df["seed"]).astype(int).unique().tolist()),
    "scoring_versions": sorted(ranking_df["scoring_version"].astype(str).unique().tolist()),
}
review_experiment_id = hashlib.sha256(
    json.dumps(review_payload, sort_keys=True).encode()
).hexdigest()[:12]

if not REVIEW_PATH.exists():
    review_parts = []
    for spec_id, samples in sample_stats.groupby("spec_id"):
        samples = samples.sort_values("selection_score")
        count = min(MANUAL_SAMPLES_PER_SPEC, len(samples))
        indices = np.rint(np.linspace(0, len(samples) - 1, count)).astype(int)
        for sample_id in dict.fromkeys(samples.iloc[indices]["sample_id"]):
            candidates = ranking_df[
                (ranking_df["sample_id"] == sample_id)
                & ~ranking_df["failure_reasons"].astype("string").str.contains(
                    "generation_failed", na=False
                )
            ].sort_values("auto_rank")
            if candidates.empty:
                continue
            review_parts.append(candidates.iloc[[0, len(candidates) // 2]])

    if not review_parts:
        raise RuntimeError("Không có candidate hợp lệ để tạo manual review subset.")

    review_df = pd.concat(review_parts, ignore_index=True).drop_duplicates(["sample_id", "seed"])
    review_df["review_experiment_id"] = review_experiment_id
    review_df = review_df[[
        "review_experiment_id", "sample_id", "spec_id", "heap_rank",
        "selection_score", "context_type", "seed", "output_path",
        "scoring_version",
        "auto_rank", "hard_valid", "soft_score", "failure_reasons",
    ]].copy()
    for column in [
        "human_valid", "human_anatomy", "human_pose_contact",
        "human_context_preservation", "human_photorealism",
        "human_accept", "human_notes",
    ]:
        review_df[column] = ""
    review_df.to_csv(REVIEW_PATH, index=False, sep=";", encoding="utf-8-sig")
    print("Created:", REVIEW_PATH)
else:
    review_df = pd.read_csv(REVIEW_PATH, sep=";", encoding="utf-8-sig")
    stored_ids = set(
        review_df.get("review_experiment_id", pd.Series(dtype=str)).dropna().astype(str)
    )
    if stored_ids != {review_experiment_id}:
        raise RuntimeError(
            "manual_review_subset.csv không khớp experiment hiện tại; "
            "hãy đổi tên file cũ trước khi tạo subset mới."
        )
    print("Using existing:", REVIEW_PATH)

display(review_df.head())


## 3. Kiểm định automatic evaluator

Sau khi điền các cột `human_*` trong CSV, chạy lại cell này. Kết quả gồm precision/recall/agreement của hard gate và Pearson/Spearman giữa soft score hoặc candidate-selection score với đánh giá thủ công.


In [ ]:
def correlation_rows(frame, predictor, outcomes, source):
    rows = []
    for outcome in outcomes:
        valid = frame[[predictor, outcome]].dropna()
        if len(valid) < 3 or valid[predictor].nunique() < 2 or valid[outcome].nunique() < 2:
            pearson = pearson_p = spearman = spearman_p = np.nan
        else:
            pearson, pearson_p = stats.pearsonr(valid[predictor], valid[outcome])
            spearman, spearman_p = stats.spearmanr(valid[predictor], valid[outcome])
        rows.append({
            "source": source, "predictor": predictor, "outcome": outcome,
            "n": len(valid), "pearson_r": pearson, "pearson_p": pearson_p,
            "spearman_rho": spearman, "spearman_p": spearman_p,
        })
    return rows


reviewed = review_df[pd.to_numeric(review_df["human_accept"], errors="coerce").notna()].copy()
if len(reviewed) < MIN_REVIEWED_ROWS:
    print(f"Cần ít nhất {MIN_REVIEWED_ROWS} dòng đã chấm; hiện có {len(reviewed)}.")
else:
    binary_columns = ["human_valid", "human_accept"]
    quality_columns = [
        "human_anatomy", "human_pose_contact",
        "human_context_preservation", "human_photorealism",
    ]
    for column in binary_columns + quality_columns:
        reviewed[column] = pd.to_numeric(reviewed[column], errors="coerce")
    reviewed["human_mean_quality"] = reviewed[quality_columns].mean(axis=1) / 5.0

    ranking_validation = pd.DataFrame(correlation_rows(
        reviewed, "soft_score", ["human_accept", "human_mean_quality"], "manual_review"
    ))
    ranking_validation.to_csv(
        OUTPUT_DIR / "automatic_human_validation.csv",
        index=False, sep=";", encoding="utf-8-sig",
    )

    gate_reviewed = reviewed.dropna(subset=["human_valid"])
    predicted = gate_reviewed["hard_valid"].astype(int)
    actual = gate_reviewed["human_valid"].astype(int)
    tp = int(((predicted == 1) & (actual == 1)).sum())
    fp = int(((predicted == 1) & (actual == 0)).sum())
    fn = int(((predicted == 0) & (actual == 1)).sum())
    gate_validation = pd.DataFrame([{
        "n": len(gate_reviewed),
        "precision": tp / (tp + fp) if tp + fp else np.nan,
        "recall": tp / (tp + fn) if tp + fn else np.nan,
        "agreement": float((predicted == actual).mean()) if len(actual) else np.nan,
    }])
    gate_validation.to_csv(
        OUTPUT_DIR / "hard_gate_human_validation.csv",
        index=False, sep=";", encoding="utf-8-sig",
    )

    human_by_sample = reviewed.groupby("sample_id").agg(
        human_success_rate=("human_accept", "mean")
    ).reset_index()
    human_selection = sample_stats.merge(human_by_sample, on="sample_id", how="inner")
    selection_rows = correlation_rows(
        human_selection, "selection_score", ["human_success_rate"], "manual_review"
    )
    human_selection["selection_score_percentile"] = human_selection.groupby("spec_id")[
        "selection_score"
    ].rank(method="average", pct=True)
    selection_rows.extend(correlation_rows(
        human_selection,
        "selection_score_percentile",
        ["human_success_rate"],
        "manual_review_within_spec_normalized",
    ))
    selection_validation = pd.DataFrame(selection_rows)
    selection_validation.to_csv(
        OUTPUT_DIR / "manual_selection_correlations.csv",
        index=False, sep=";", encoding="utf-8-sig",
    )

    display(gate_validation)
    display(ranking_validation)
    display(selection_validation)
